In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader
from torchvision import transforms

# Make the repository root importable when this notebook is opened from notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.dataset import TripleImageDataset
from models.fusion_model import FusionLateModel

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 224
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

print(f"Repository: {REPO_ROOT}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("CUDA is unavailable; using CPU fallback.")

Repository: /home/nouran/MMFM
Device: cuda
GPU: NVIDIA GeForce RTX 3060
CUDA version: 13.0


In [2]:
# Build a tiny on-disk manifest with three image modalities.
# Some rows intentionally omit one modality to exercise presence_mask.
DATA_DIR = REPO_ROOT / "data" / "quick_start_demo"
DATA_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for index, label in enumerate([0, 1, 0, 1, 0, 1]):
    image_paths = []
    for modality in range(3):
        path = DATA_DIR / f"sample_{index}_modality_{modality + 1}.png"
        image = Image.new("RGB", (image_size, image_size), (30 + label * 180, 40 + modality * 50, 120))
        draw = ImageDraw.Draw(image)
        draw.rectangle((8 + modality * 4, 8, 24 + modality * 4, 24), fill=(240, 240, 240))
        draw.text((8, 42), f"{label}/{modality + 1}", fill=(255, 255, 255))
        image.save(path)
        image_paths.append(str(path))

    if index == 1:
        image_paths[1] = "MISSING"
    elif index == 2:
        image_paths[2] = "MISSING"
    elif index == 4:
        image_paths[0] = "MISSING"
    rows.append({"img1": image_paths[0], "img2": image_paths[1], "img3": image_paths[2], "label": label})

manifest_path = DATA_DIR / "manifest.csv"
pd.DataFrame(rows).to_csv(manifest_path, index=False)

dataset = TripleImageDataset(
    manifest_path,
    transform1=transform,
    transform2=transform,
    transform3=transform,
)
loader = DataLoader(dataset, batch_size=3, shuffle=False)

batch = next(iter(loader))
images = batch[:3]
labels = batch[3]
presence = batch[4]
print(f"Samples: {len(dataset)}")
print(f"Batch image shape: {tuple(images[0].shape)}")
print(f"Labels: {labels.tolist()}")
print(f"Presence masks: {presence.tolist()}")

Samples: 6
Batch image shape: (3, 3, 224, 224)
Labels: [0, 1, 0]
Presence masks: [[True, True, True], [True, False, True], [True, True, False]]


In [7]:
# Use the built-in fallback CNN for a fast, offline smoke test.
model = FusionLateModel(
    backbone_names=("simple", "simple", "simple"),
    pretrained=False,
    embedding_dim=16,
    num_classes=2,
    fusion_mode="masked_scalar",
).to(device)

model_device = next(model.parameters()).device
assert model_device.type == device.type, f"Model is on {model_device}, expected {device}"

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for epoch in range(2):
    epoch_loss = 0.0
    for x1, x2, x3, labels, presence_mask in loader:
        x1, x2, x3 = x1.to(device), x2.to(device), x3.to(device)
        labels, presence_mask = labels.to(device), presence_mask.to(device)
        assert x1.device.type == device.type and labels.device.type == device.type

        optimizer.zero_grad()
        logits, branch_logits, weights = model(x1, x2, x3, presence_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/2 - loss: {epoch_loss / len(loader):.4f}")

print(f"Model and training batch successfully used: {device}")

Epoch 1/2 - loss: 0.6904
Epoch 2/2 - loss: 0.6738
Model and training batch successfully used: cuda


In [4]:
# Run inference on one batch and inspect the learned fusion weights.
model.eval()
with torch.no_grad():
    x1, x2, x3, labels, presence_mask = next(iter(loader))
    logits, branch_logits, weights = model(
        x1.to(device),
        x2.to(device),
        x3.to(device),
        presence_mask.to(device),
    )
    predictions = logits.argmax(dim=1)

results = pd.DataFrame({
    "label": labels.tolist(),
    "prediction": predictions.cpu().tolist(),
    "confidence": logits.softmax(dim=1).max(dim=1).values.cpu().round(decimals=3).tolist(),
})
print(results)
print("Fusion weights for the batch:")
print(weights.cpu().round(decimals=3))

   label  prediction  confidence
0      0           1       0.520
1      1           1       0.540
2      0           1       0.543
Fusion weights for the batch:
tensor([[0.3330, 0.3330, 0.3330],
        [0.5000, 0.0000, 0.5000],
        [0.5000, 0.5000, 0.0000]])
